# Camp Echo-Chamber Metrics (Giant-Component Restricted)

Split out of `structural-backfire-analysis.ipynb` / `structural-3arm-dynamics.ipynb`
(both had grown too large). This notebook does **no GEXF parsing** — it loads
`results/BF_analysis/structural_camp_3arm.csv`, produced by
`structural-3arm-dynamics.ipynb`'s collection cell, which already includes the
giant-component metrics below (`compute_camp_giant_metrics` in `bf_common.py`).
**Run that notebook first** if the CSV is missing or stale.

For each direction-relative camp, at each GEXF snapshot, we take the camp-induced
subgraph and restrict to **its own giant component** (same camp-first split as
`compute_camp_structural`'s `camp_giant_frac`), then compute:

| Metric | Meaning |
|---|---|
| `gc_frac` | fraction of the camp inside its own giant component — **coverage diagnostic**: if low, the other metrics below describe only a minority of the camp and should be read cautiously |
| `gc_density` | edge density within the giant component |
| `gc_clustering_local` | mean per-node (local) clustering coefficient |
| `gc_clustering_global` | transitivity (global clustering coefficient) |
| `gc_modularity` | Louvain modularity *within* the giant component — does the camp itself further split into sub-cliques? |
| `gc_path_length` | average shortest path length (well-defined since the giant component is connected by construction) |
| `gc_ei_index` | E-I homophily index, restricted to giant-component members (+1 hetero / −1 echo) |
| `gc_reciprocity` | fraction of mutual-follow pairs (directed-graph-specific) |
| `gc_disagreement` | Σ_{i<j} (op_i − op_j)² over giant-component opinions (== n² × population variance) |
| `gc_opinion_var` | population variance of opinion within the giant component |

All plots follow the same manip(pooled dir) vs. control overlay convention as
Section 3.1/3.2, with a vertical line at the manipulation onset (`ONSET_STEP`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
from bf_common import *

_camp_csv = os.path.join(SUMMARY_DIR, "structural_camp_3arm.csv")
if not os.path.exists(_camp_csv):
    raise FileNotFoundError(
        f"{_camp_csv} not found — run structural-3arm-dynamics.ipynb's collection "
        "cell first to generate it."
    )
df_camp = pd.read_csv(_camp_csv)
print(f"Loaded df_camp: {df_camp.shape}  (mtime-check: re-run structural-3arm-dynamics.ipynb "
      "if this predates your latest simulation results)")


### Coverage diagnostic: how much of each camp does the giant component cover?

If `gc_frac` is low for a camp (e.g. the camp is fragmented into several similarly-sized
components rather than one dominant one), the giant-component-restricted metrics below
describe only a minority of that camp's agents — treat them as illustrative, not
representative, for that camp/step/arm, and consider inspecting the full connected-component
size distribution directly (re-parsing the relevant GEXF) if it matters for the conclusion.

In [ ]:
# ---------------------------------------------------------------------------
# gc_frac summary: mean and min per camp x arm, flagging low-coverage cases
# ---------------------------------------------------------------------------
LOW_COVERAGE_THRESHOLD = 0.7

frac_summary = (df_camp.groupby(['camp', 'arm'])['gc_frac']
                .agg(['mean', 'min', 'count'])
                .reindex(GROUP_ORDER, level='camp')
                .round(3))
print("gc_frac summary by camp x arm (mean / min / n observations):")
print(frac_summary.to_string())

low = frac_summary[frac_summary['mean'] < LOW_COVERAGE_THRESHOLD]
if low.empty:
    print(f"\nNo camp/arm has mean gc_frac below {LOW_COVERAGE_THRESHOLD} — giant component "
          "is a reasonably representative summary throughout.")
else:
    print(f"\n[!] camp/arm combos with mean gc_frac < {LOW_COVERAGE_THRESHOLD} "
          "(interpret gc_* metrics cautiously here):")
    print(low.to_string())


In [ ]:
# ---------------------------------------------------------------------------
# gc_frac time series per camp: manip (pooled dir) vs control
# ---------------------------------------------------------------------------
def camp_series(camp, mask, col):
    sub = df_camp[(df_camp['camp'] == camp) & mask]
    return sub.groupby('step')[col].agg(['mean', 'sem'])

def plot_camp_metric(col, title, ylim=None):
    m_mask = (df_camp['arm'] == 'manip')
    c_mask = (df_camp['arm'] == 'control')
    fig, axes = plt.subplots(1, len(GROUP_ORDER), figsize=(4.2 * len(GROUP_ORDER), 4), sharey=True)
    for ax, camp in zip(axes, GROUP_ORDER):
        for lbl, mask, color in [('manip', m_mask, '#c0392b'), ('control', c_mask, '#2c3e50')]:
            g = camp_series(camp, mask, col)
            if g.empty:
                continue
            ax.plot(g.index, g['mean'], label=lbl, color=color, lw=2)
            ax.fill_between(g.index, g['mean'] - g['sem'], g['mean'] + g['sem'],
                            color=color, alpha=0.15)
        ax.axvline(ONSET_STEP, color='k', ls='--', lw=1, alpha=0.6)
        ax.set_title(camp, fontsize=11, color=CAMP_COLORS.get(camp, 'k'))
        ax.set_xlabel('step'); ax.grid(alpha=0.3)
        if ylim is not None:
            ax.set_ylim(*ylim)
    axes[0].set_ylabel(title, fontsize=11)
    axes[0].legend(fontsize=9)
    fig.suptitle(title, fontsize=14)
    fig.tight_layout(); plt.show()

plot_camp_metric('gc_frac', 'Giant-component fraction of camp', ylim=(0, 1))


### Giant-component structure and echo-chamber metrics, per camp

In [ ]:
# ---------------------------------------------------------------------------
# Full metric x camp grid (manip vs control): structural echo-chamber metrics
# ---------------------------------------------------------------------------
GC_STRUCT_METRICS = [
    ('gc_density',           'GC density'),
    ('gc_clustering_local',  'GC local clustering (avg)'),
    ('gc_clustering_global', 'GC global clustering (transitivity)'),
    ('gc_modularity',        'GC modularity Q (sub-structure within camp)'),
    ('gc_path_length',       'GC avg shortest path length'),
]

def plot_metric_grid(metrics, suptitle):
    fig, axes = plt.subplots(len(metrics), len(GROUP_ORDER),
                             figsize=(4.2 * len(GROUP_ORDER), 3.2 * len(metrics)),
                             sharex=True)
    if len(metrics) == 1:
        axes = axes[None, :]
    for r, (col, title) in enumerate(metrics):
        for c, camp in enumerate(GROUP_ORDER):
            ax = axes[r, c]
            for lbl, mask, color in [('manip', df_camp['arm'] == 'manip', '#c0392b'),
                                     ('control', df_camp['arm'] == 'control', '#2c3e50')]:
                g = camp_series(camp, mask, col)
                if g.empty:
                    continue
                ax.plot(g.index, g['mean'], color=color, lw=1.6, label=lbl)
                ax.fill_between(g.index, g['mean'] - g['sem'], g['mean'] + g['sem'],
                                color=color, alpha=0.13)
            ax.axvline(ONSET_STEP, color='k', ls='--', lw=0.8, alpha=0.5)
            ax.grid(alpha=0.3)
            if r == 0:
                ax.set_title(camp, fontsize=11, color=CAMP_COLORS.get(camp, 'k'))
            if c == 0:
                ax.set_ylabel(title, fontsize=9)
    axes[0, 0].legend(fontsize=8)
    fig.suptitle(suptitle, fontsize=14)
    fig.tight_layout(); plt.show()

plot_metric_grid(GC_STRUCT_METRICS, 'Giant-component structural metrics: manip (red) vs control (black)')


In [ ]:
# ---------------------------------------------------------------------------
# Homophily / directional metrics: E-I index and reciprocity
# ---------------------------------------------------------------------------
GC_HOMOPHILY_METRICS = [
    ('gc_ei_index',    'GC E-I index (+1 hetero / -1 echo)'),
    ('gc_reciprocity', 'GC reciprocity (mutual-follow fraction)'),
]
plot_metric_grid(GC_HOMOPHILY_METRICS, 'Giant-component homophily metrics: manip (red) vs control (black)')


In [ ]:
# ---------------------------------------------------------------------------
# Opinion disagreement within the giant component
# ---------------------------------------------------------------------------
GC_OPINION_METRICS = [
    ('gc_disagreement', 'GC disagreement  (Σ_i<j (op_i-op_j)²)'),
    ('gc_opinion_var',  'GC opinion variance'),
]
plot_metric_grid(GC_OPINION_METRICS, 'Giant-component opinion disagreement: manip (red) vs control (black)')


### Onset → end change, by camp and arm

Same convention as Table 3.3/3.4: `delta = value at step 40000 minus value at the onset
step`, averaged over seed×direction units. `manip - control` isolates the
manipulation-attributable change per camp, controlling for whatever the no-target
counterfactual does on its own.

In [ ]:
# ---------------------------------------------------------------------------
# Table: onset->end change (ONSET_STEP -> 40000) for all gc_* metrics, by camp and arm
# ---------------------------------------------------------------------------
GC_ALL_METRICS = GC_STRUCT_METRICS + GC_HOMOPHILY_METRICS + GC_OPINION_METRICS + [('gc_frac', 'GC coverage fraction')]

def gc_onset_end_delta(col):
    piv = (df_camp[df_camp['step'].isin([ONSET_STEP, 40000])]
           .pivot_table(index=['arm', 'camp'], columns='step', values=col, aggfunc='mean'))
    piv['delta'] = piv.get(40000) - piv.get(ONSET_STEP)
    out = piv['delta'].unstack('arm')
    if 'manip' in out and 'control' in out:
        out['manip - control'] = out['manip'] - out['control']
    return out.reindex(GROUP_ORDER)

for col, title in GC_ALL_METRICS:
    print(f"\n=== {title}  ({col}): onset->end delta ===")
    print(gc_onset_end_delta(col).round(4))
